#### Simple Gen AI APP using Langchain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPEN_API_KEY'] = os.getenv("OPENAI_API_KEY")
## Langsmith tracking
os.environ['LANGSMITH_ENDPOINT'] = "https://apac.api.smith.langchain.com"
os.environ['LANGSMITH_API_KEY'] = os.getenv("LANGSMITH_API_KEY")
os.environ['LANGSMITH_TRACING'] = "true"
os.environ['LANGSMITH_PROJECT'] = os.getenv("LANGSMITH_PROJECT")

## Load Data -> Doc -> Divide our text into chunks -> text -> vectors -> Vector Embeddings -> Vector Store

In [4]:
## Data ingestion - From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader


In [5]:
loader = WebBaseLoader("https://www.datacamp.com/tutorial/introduction-to-langsmith")
loader

In [6]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://www.datacamp.com/tutorial/introduction-to-langsmith', 'title': 'An Introduction to Debugging And Testing LLMs in LangSmith | DataCamp', 'description': 'Discover how LangSmith optimizes LLM testing and debugging for AI applications. Enhance quality assurance and streamline development with real-world examples.\n', 'language': 'en'}, page_content='An Introduction to Debugging And Testing LLMs in LangSmith | DataCampSkip to main contentENEnglishEspañolPortuguêsDeutschBetaFrançaisBetaItalianoBetaTürkçeBetaBahasa IndonesiaBetaTiếng ViệtBetaNederlandsBetaहिन्दीBeta日本語Beta한국어BetaPolskiBetaRomânăBetaРусскийBetaSvenskaBetaไทยBeta中文(简体)BetaMore InformationTutorialsBlogsTutorialsdocsPodcastsCheat Sheetscode-alongsNewsletterCategoryCategory Technologies Discover content by tools and technologyAI AgentsAI NewsArtificial IntelligenceAWSAzureBusiness IntelligenceChatGPTDatabricksdbtDockerExcelGenerative AIGitGoogle Cloud PlatformHugging FaceJavaJuliaKafkaKuberne

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)

In [8]:
documents

[Document(metadata={'source': 'https://www.datacamp.com/tutorial/introduction-to-langsmith', 'title': 'An Introduction to Debugging And Testing LLMs in LangSmith | DataCamp', 'description': 'Discover how LangSmith optimizes LLM testing and debugging for AI applications. Enhance quality assurance and streamline development with real-world examples.\n', 'language': 'en'}, page_content='An Introduction to Debugging And Testing LLMs in LangSmith | DataCampSkip to main contentENEnglishEspañolPortuguêsDeutschBetaFrançaisBetaItalianoBetaTürkçeBetaBahasa IndonesiaBetaTiếng ViệtBetaNederlandsBetaहिन्दीBeta日本語Beta한국어BetaPolskiBetaRomânăBetaРусскийBetaSvenskaBetaไทยBeta中文(简体)BetaMore InformationTutorialsBlogsTutorialsdocsPodcastsCheat Sheetscode-alongsNewsletterCategoryCategory Technologies Discover content by tools and technologyAI AgentsAI NewsArtificial IntelligenceAWSAzureBusiness IntelligenceChatGPTDatabricksdbtDockerExcelGenerative AIGitGoogle Cloud PlatformHugging FaceJavaJuliaKafkaKuberne

In [9]:
from langchain_ollama import OllamaEmbeddings
embeddings=(
    OllamaEmbeddings(
        # model="gemma:2b"
        model="nomic-embed-text"
        )
)

In [11]:
from langchain_community.vectorstores import FAISS
vectorstoredb = FAISS.from_documents(documents, embeddings)

In [12]:
vectorstoredb

In [14]:
# Query from a Vector DB
query = "To pass a custom criteria with natural language, we simply pass"
result = vectorstoredb.similarity_search(query)
result[0].page_content

'To pass a custom criteria with natural language, we simply pass {“criteria_name”: “Condition to check”} to the Criteria class. Above, we are creating two extra evaluators, so LangSmith will run two additional prompts on top of the output produced by the prompts in our dataset:\n# Run the evaluation\nresults = run_on_dataset(\n   client,\n   dataset_name,\n   llm,\n   evaluation=eval_config,\n   project_name="custom_criteria_test",\n)\n\nIf you check out the run, you will see the custom criteria we’ve defined under each prompt. If you hover over, you will get the reasoning of the LLM after:'

In [15]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

In [23]:
## Retrieval Chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""
Answer the following question based only on the provided context:
<context>
{context}
</context>

Question:
{input}
"""
)

document_chain = create_stuff_documents_chain(llm ,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\nQuestion:\n{input}\n'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, model='llama3.2:1b', temperature=0.0)
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [25]:
from langchain_core.documents import Document
document_chain.invoke({
    "input": "Tell me about Real-time monitoring and visualization in Langsmith",
    "context": [Document(page_content="LangSmith uses traces to log almost every aspect of LLM runs. These include metrics such as latency, token count, price of runs, and all types of metadata. The Web UI allows you to quickly filter runs based on error percentage, latency, date, or even by text content using natural language. This means that if, for instance, an AI tutor starts glitching in its responses to actual students, you can push out a fix in a few hours. Another great feature of LangSmith is datasets. They can be used to improve LangChain chains, agents or models against a set of standardized examples before deployment. For example, we may have a CSV file containing two columns — questions and answers for flashcards in a specific format. By converting this file into a reference dataset, we can instruct LLMs to evaluate their own output using the quality assurance metrics mentioned earlier.")]
})

"Based on the provided context, here's what I've learned about real-time monitoring and visualization in Langsmith:\n\nLangSmith uses traces to log almost every aspect of LLM (Large Language Model) runs. This includes metrics such as latency, token count, price of runs, and all types of metadata.\n\nReal-time monitoring is enabled through the Web UI, which allows users to quickly filter runs based on error percentage, latency, date, or even by text content using natural language. This enables quick identification and resolution of issues in LLMs.\n\nAdditionally, LangSmith provides datasets that can be used to improve LangChain chains, agents, or models against a set of standardized examples before deployment. By converting these datasets into reference datasets, users can evaluate the quality assurance metrics mentioned earlier (e.g., error percentage, latency) using real-time monitoring and visualization tools in Langsmith.\n\nIn summary, Langsmith offers real-time monitoring and vis

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question. 


A Retriever in LangChain is a component responsible for fetching the most relevant documents from a data source (such as a vector database) based on a user's query. It acts as the bridge between your knowledge base and the LLM by returning relevant Document objects, which are then used as context to generate accurate, grounded responses in a RAG pipeline.

In [26]:
### Input -> Retreiver -> Vectorstoredb

vectorstoredb

In [31]:
retreiver = vectorstoredb.as_retriever()

from langchain_classic.chains import create_retrieval_chain

retreival_chain = create_retrieval_chain(retreiver, document_chain)

In [32]:
retreival_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000022115347530>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\nQuestion:\n{input}\n'), ad

In [34]:
response = retreival_chain.invoke({"input": "Tell me about Real-time monitoring and visualization in Langsmith"})
response['answer']

"Real-time monitoring and visualization is a critical component of LangSmith, which allows users to quickly filter runs based on various metrics such as latency, token count, price of runs, and metadata.\n\nWith LangSmith, you can use the Web UI to monitor your LLM runs in real-time. This enables you to:\n\n* Filter runs by error percentage\n* Filter runs by latency\n* Filter runs by date\n* Filter runs by text content\n\nThis means that if an AI tutor starts glitching in its responses to actual students, you can quickly push out a fix in a few hours.\n\nLangSmith also provides real-time visualization capabilities, which allows users to see the performance of their LLMs in real-time. This enables them to:\n\n* Track key metrics such as latency and token count\n* Visualize the output of their LLMs over time\n\nBy providing both real-time monitoring and visualization capabilities, LangSmith enables users to quickly identify issues with their LLMs and make data-driven decisions to improve